In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import re
import string
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()

def word_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(rf"[{re.escape(string.punctuation)}]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["len1_words"] = df["sentence1"].map(word_count)
df["len2_words"] = df["sentence2"].map(word_count)
df["avg_words"] = (df["len1_words"] + df["len2_words"]) / 2.0
df["len_gap_words"] = (df["len1_words"] - df["len2_words"]).abs()

filtered_df = df[
    df["len1_words"].between(7, 18)
    & df["len2_words"].between(7, 18)
    & df["len_gap_words"] <= 6
].copy()

subset_size = min(300, len(filtered_df))
subset_df = filtered_df.sort_values(
    by=["avg_words", "len_gap_words", "sentence1", "sentence2"],
    ascending=[True, True, True, True],
).head(subset_size).reset_index(drop=True)

subset_df["sentence1_clean"] = subset_df["sentence1"].map(clean_text)
subset_df["sentence2_clean"] = subset_df["sentence2"].map(clean_text)
subset_df["sentence1_changed"] = subset_df["sentence1"] != subset_df["sentence1_clean"]
subset_df["sentence2_changed"] = subset_df["sentence2"] != subset_df["sentence2_clean"]
subset_df["either_changed"] = subset_df["sentence1_changed"] | subset_df["sentence2_changed"]
subset_df["sentence1_char_delta"] = subset_df["sentence1"].str.len() - subset_df["sentence1_clean"].str.len()
subset_df["sentence2_char_delta"] = subset_df["sentence2"].str.len() - subset_df["sentence2_clean"].str.len()

print({
    "original_num_examples": len(df),
    "filtered_num_examples": len(filtered_df),
    "subset_num_examples": len(subset_df),
    "num_sentence1_changed": int(subset_df["sentence1_changed"].sum()),
    "num_sentence2_changed": int(subset_df["sentence2_changed"].sum()),
    "num_either_changed": int(subset_df["either_changed"].sum()),
    "columns": subset_df.columns.tolist(),
})
print(subset_df[["sentence1", "sentence1_clean", "sentence2", "sentence2_clean", "label", "sentence1_changed", "sentence2_changed"]].head(10))


In [ ]:
length_stats = {
    "len1_words_mean": float(subset_df["len1_words"].mean()),
    "len1_words_median": float(subset_df["len1_words"].median()),
    "len1_words_min": int(subset_df["len1_words"].min()),
    "len1_words_max": int(subset_df["len1_words"].max()),
    "len2_words_mean": float(subset_df["len2_words"].mean()),
    "len2_words_median": float(subset_df["len2_words"].median()),
    "len2_words_min": int(subset_df["len2_words"].min()),
    "len2_words_max": int(subset_df["len2_words"].max()),
    "avg_words_mean": float(subset_df["avg_words"].mean()),
    "avg_words_median": float(subset_df["avg_words"].median()),
    "len_gap_words_mean": float(subset_df["len_gap_words"].mean()),
    "len_gap_words_median": float(subset_df["len_gap_words"].median()),
    "pct_either_changed": float(subset_df["either_changed"].mean()),
    "mean_sentence1_char_delta": float(subset_df["sentence1_char_delta"].mean()),
    "mean_sentence2_char_delta": float(subset_df["sentence2_char_delta"].mean()),
}
print(length_stats)


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)


In [ ]:
sentences1_raw = subset_df["sentence1"].tolist()
sentences2_raw = subset_df["sentence2"].tolist()
sentences1_clean = subset_df["sentence1_clean"].tolist()
sentences2_clean = subset_df["sentence2_clean"].tolist()
labels = subset_df["label"].to_numpy(dtype=np.float32)

emb1_raw = model.encode(
    sentences1_raw,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2_raw = model.encode(
    sentences2_raw,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb1_clean = model.encode(
    sentences1_clean,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

emb2_clean = model.encode(
    sentences2_clean,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

cosine_similarity_raw = np.sum(emb1_raw * emb2_raw, axis=1)
predicted_score_raw_0_5 = 2.5 * (cosine_similarity_raw + 1.0)

cosine_similarity_clean = np.sum(emb1_clean * emb2_clean, axis=1)
predicted_score_clean_0_5 = 2.5 * (cosine_similarity_clean + 1.0)


In [ ]:
pearson_raw = pearsonr(predicted_score_raw_0_5, labels).statistic
spearman_raw = spearmanr(predicted_score_raw_0_5, labels).statistic
mae_raw = float(np.mean(np.abs(predicted_score_raw_0_5 - labels)))

pearson_clean = pearsonr(predicted_score_clean_0_5, labels).statistic
spearman_clean = spearmanr(predicted_score_clean_0_5, labels).statistic
mae_clean = float(np.mean(np.abs(predicted_score_clean_0_5 - labels)))

results_df = subset_df.copy()
results_df["cosine_similarity_raw"] = cosine_similarity_raw
results_df["predicted_score_raw_0_5"] = predicted_score_raw_0_5
results_df["absolute_error_raw"] = np.abs(results_df["predicted_score_raw_0_5"] - results_df["label"])
results_df["cosine_similarity_clean"] = cosine_similarity_clean
results_df["predicted_score_clean_0_5"] = predicted_score_clean_0_5
results_df["absolute_error_clean"] = np.abs(results_df["predicted_score_clean_0_5"] - results_df["label"])
results_df["prediction_delta"] = results_df["predicted_score_clean_0_5"] - results_df["predicted_score_raw_0_5"]
results_df["absolute_prediction_delta"] = results_df["prediction_delta"].abs()
results_df["absolute_error_delta"] = results_df["absolute_error_clean"] - results_df["absolute_error_raw"]

largest_prediction_change_df = results_df.sort_values(
    ["absolute_prediction_delta", "absolute_error_clean"], ascending=[False, False]
).reset_index(drop=True)

largest_clean_errors_df = results_df.sort_values("absolute_error_clean", ascending=False).reset_index(drop=True)

print(results_df[[
    "sentence1", "sentence1_clean", "sentence2", "sentence2_clean", "label",
    "predicted_score_raw_0_5", "predicted_score_clean_0_5", "prediction_delta",
    "absolute_error_raw", "absolute_error_clean", "sentence1_changed", "sentence2_changed"
]].head(10))

print(largest_prediction_change_df[[
    "sentence1", "sentence1_clean", "sentence2", "sentence2_clean", "label",
    "predicted_score_raw_0_5", "predicted_score_clean_0_5", "absolute_prediction_delta",
    "absolute_error_raw", "absolute_error_clean", "sentence1_changed", "sentence2_changed"
]].head(10))


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"original_num_examples: {len(df)}")
print(f"filtered_num_examples: {len(filtered_df)}")
print(f"subset_num_examples: {len(subset_df)}")
print(f"subset_rule: sentence1_words_and_sentence2_words_in_[7,18]_and_length_gap<=6_then_sorted_deterministically_head_{len(subset_df)}")
print(f"num_sentence1_changed: {int(results_df['sentence1_changed'].sum())}")
print(f"num_sentence2_changed: {int(results_df['sentence2_changed'].sum())}")
print(f"num_either_changed: {int(results_df['either_changed'].sum())}")
print(f"pearson_raw: {pearson_raw:.6f}")
print(f"spearman_raw: {spearman_raw:.6f}")
print(f"mae_raw: {mae_raw:.6f}")
print(f"pearson_clean: {pearson_clean:.6f}")
print(f"spearman_clean: {spearman_clean:.6f}")
print(f"mae_clean: {mae_clean:.6f}")
print(f"mean_absolute_prediction_delta: {results_df['absolute_prediction_delta'].mean():.6f}")
print(f"max_absolute_prediction_delta: {results_df['absolute_prediction_delta'].max():.6f}")
print(f"mean_absolute_error_delta_clean_minus_raw: {results_df['absolute_error_delta'].mean():.6f}")
print(f"avg_len1_words: {subset_df['len1_words'].mean():.2f}")
print(f"avg_len2_words: {subset_df['len2_words'].mean():.2f}")
print(f"avg_length_gap_words: {subset_df['len_gap_words'].mean():.2f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")

top_changed_examples = largest_prediction_change_df[[
    "sentence1", "sentence1_clean", "sentence2", "sentence2_clean", "label",
    "predicted_score_raw_0_5", "predicted_score_clean_0_5", "prediction_delta",
    "absolute_prediction_delta", "absolute_error_raw", "absolute_error_clean",
    "sentence1_changed", "sentence2_changed", "sentence1_char_delta", "sentence2_char_delta"
]].head(5)
print(top_changed_examples.to_dict(orient="records"))
